# Imports

In [ ]:
import requests
import geopandas
import pyarrow

import duckdb
from lonboard import Map, PolygonLayer
from lonboard.basemap import CartoStyle

import warnings
warnings.filterwarnings('ignore')

# Initialize DuckDB

In [ ]:
con = duckdb.connect(database = ':memory:')
#con.execute("INSTALL spatial; INSTALL httpfs")
con.execute("LOAD spatial; LOAD httpfs")

# Data Source 1

https://source.coop/vida/google-microsoft-osm-open-buildings

## Select Partitioning

In [ ]:
iso = "LSO" # LESOTO

In [ ]:
url = f"https://data.source.coop/vida/google-microsoft-osm-open-buildings/geoparquet/by_country/country_iso={iso}/{iso}.parquet"

## Get Data

In [ ]:
con.execute(
    f"""
    CREATE TABLE lesoto_osm AS
    SELECT *
    FROM read_parquet(
        '{url}'    
    )
    """
)

In [ ]:
con.sql(
    f"""
    DESCRIBE lesoto_osm
    """
)

In [ ]:
con.sql(
    f"""
    SELECT *
    FROM lesoto_osm
    LIMIT 5
    """
)

In [ ]:
con.sql(
    f"""
    SELECT DISTINCT bf_source
    FROM lesoto_osm
    """
)

In [ ]:
con.sql(
    f'''
    SELECT bf_source as data_source, count(*) as count
    FROM lesoto_osm
    GROUP BY data_source
    ORDER BY count ASC
    '''
)

# Data Source 2

https://source.coop/vida/google-microsoft-open-buildings

## Select Partitioning

In [ ]:
url = f"https://data.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=LSO/2017612633061982208.parquet"

## Get Data

In [ ]:
con.execute(
    f"""
    CREATE TABLE lesoto_google AS
    SELECT *
    FROM read_parquet(
        '{url}'
    )
    """
)

In [ ]:
con.sql(
    f"""
    DESCRIBE lesoto_google
    """
)

In [ ]:
con.sql(
    f"""
    SELECT *
    FROM lesoto_google
    LIMIT 5
    """
)

In [ ]:
con.sql(
    f"""
    SELECT DISTINCT bf_source
    FROM lesoto_google
    """
)

In [ ]:
con.sql(
    f'''
    SELECT bf_source as data_source, count(*) as count
    FROM lesoto_google
    GROUP BY data_source
    ORDER BY count ASC
    '''
)

# Register Count

In [ ]:
con.sql(
    f"""
    SELECT COUNT(*) AS count
    FROM lesoto_osm
    WHERE bf_source = 'google'
    """
)

In [ ]:
con.sql(
    f"""
    SELECT COUNT(*) AS count
    FROM lesoto_google
    WHERE bf_source = 'google'
    """
)

# Limit Area

In [ ]:
area_limit_filepath = f"data/limits.geojson"

In [ ]:
con.sql(
    f'''
    CREATE OR REPLACE TABLE area_of_interest AS
    SELECT * FROM st_read('{area_limit_filepath}')
    '''
)

In [ ]:
con.sql(
    f'''
    DESCRIBE area_of_interest
    '''
)

## Limit OSM

In [ ]:
con.sql(
    f'''
    CREATE OR REPLACE TABLE lesoto_osm_clipped AS
    SELECT bf_source , confidence, area_in_meters, s2_id, country_iso, ST_Intersection(a.geom, b.geometry) as geometry, bbox
    FROM area_of_interest as a, lesoto_osm as b
    WHERE ST_Intersects(a.geom, b.geometry);
    '''
)

In [ ]:
con.sql(
    f'''
    SELECT COUNT(*)
    FROM lesoto_osm_clipped
    '''
)

In [ ]:
con.sql(
    f'''
    SELECT *
    FROM lesoto_osm_clipped
    LIMIT 5
    '''
)

## Limit Google

In [ ]:
con.sql(
    f'''
    CREATE OR REPLACE TABLE lesoto_google_clipped AS
    SELECT bf_source, confidence, area_in_meters, s2_id, country_iso, ST_Intersection(a.geom, b.geometry) as geometry, bbox
    FROM area_of_interest as a, lesoto_google as b
    WHERE ST_Intersects(a.geom, b.geometry);
    '''
)

In [ ]:
con.sql(
    f'''
    SELECT COUNT(*)
    FROM lesoto_google_clipped
    '''
)

In [ ]:
con.sql(
    f'''
    SELECT *
    FROM lesoto_google_clipped
    LIMIT 5
    '''
)

# A U (B - I)

In [ ]:
con.sql(
    f'''
    CREATE OR REPLACE TABLE merged AS
    
    -- All from Source 1 (OSM)
    SELECT *
    FROM lesoto_osm_clipped

    UNION ALL

    -- From Source 2 (Google), only those which are not in Source 1 (OSM)
    
    SELECT *
    FROM lesoto_google_clipped AS g
    WHERE NOT EXISTS (
        SELECT 1
        FROM lesoto_osm_clipped AS o
        WHERE
            ST_Intersects(o.geometry, g.geometry)
            AND
            ST_Area(ST_Intersection(o.geometry, g.geometry)) / LEAST(ST_Area(o.geometry), ST_Area(g.geometry)) > 0.75
    )
    '''
)

In [ ]:
con.sql(
    f'''
    SELECT COUNT(*)
    FROM merged
    '''
)

In [ ]:
file_path = f"./data"

In [ ]:
con.sql(
    f'''
    COPY (SELECT * FROM merged)
    TO '{file_path}/merged.fgb'
    WITH 
        (FORMAT GDAL,
        DRIVER 'FlatGeobuf')
    '''
)

# Visualize

In [ ]:
query = f"""
    SELECT
        geometry::GEOMETRY AS geometry,
        bbox
    FROM merged
"""

In [ ]:
layer = PolygonLayer.from_duckdb(query, con, get_fill_color=[0, 0, 139])
m = Map(layer, height=500, basemap_style=CartoStyle.Positron)
m